# SciAI / SciAD Reproduction — Colab Runbook

Companion notebook to the CCC reproduction project. Run cells top to bottom.

**Before you start:** Runtime -> Change runtime type -> GPU (T4 is enough for BERT-base;
use A100/L4 if available for SciBERT-large or bigger batch sizes).

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q torch transformers sentence-transformers scikit-learn seqeval tqdm

## 3. Get the code

Two options:
- **Option A (recommended):** upload `ccc_project.zip` (from the earlier deliverable) using the file
  browser on the left, then run the cell below to unzip it.
- **Option B:** if you pushed the `code/src` files to a GitHub repo, `git clone` it instead.

In [ ]:
import os

# Option A: unzip an uploaded ccc_project.zip
if os.path.exists("ccc_project.zip"):
    !unzip -oq ccc_project.zip
    %cd ccc_project/code
else:
    print("ccc_project.zip not found in the Colab file browser.")
    print("Upload it via the Files panel on the left, or use git clone instead:")
    print('  !git clone <your-repo-url>')


## 4. Get the data

Download the official SciAI (AI task) and SciAD (AD task) train/dev/test JSON files
(see `docs/00_MASTER_GUIDE.md` section 3.3 for where to find them) and place them under
`data/`. Easiest path in Colab: upload directly, or fetch from a URL you control.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

# --- Pick ONE of the following ---

# (a) Upload manually: use the Colab file browser, drag files into ccc_project/code/data/

# (b) Download from a URL you host (uncomment and edit):
# !wget -q -O data/train.json    "https://your-host/scia i_train.json"
# !wget -q -O data/dev.json      "https://your-host/sciai_dev.json"
# !wget -q -O data/test.json     "https://your-host/sciai_test.json"
# !wget -q -O data/ad_train.json "https://your-host/sciad_train.json"
# !wget -q -O data/ad_dev.json   "https://your-host/sciad_dev.json"
# !wget -q -O data/ad_test.json  "https://your-host/sciad_test.json"

# (c) Mount Google Drive if your data already lives there:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/sciai/*.json data/

!ls -la data/


## 5. Sanity check: rule-based baseline (no GPU needed, ~seconds)

Reproduces roughly the paper's `Baseline (Rule-based)` row (P≈91.3 R≈77.9 F1≈84.1).

In [ ]:
%cd /content/ccc_project/code/src
!python rule_based_baseline.py --data ../data/test.json --out preds_ai_rule.json
!python evaluate.py --predictions preds_ai_rule.json --gold ../data/test.json --task ai


## 6. Train the AI (Acronym Identification) transformer model

Uses SciBERT by default — closest to the strong systems in the paper. Swap
`--model_name` for `bert-base-uncased` or `roberta-base` to compare encoders. Add
`--adversarial` for FGM training (the trick behind the paper's winning AT-BERT-E).

In [ ]:
!python train_ai_bert.py \
    --model_name allenai/scibert_scivocab_uncased \
    --train ../data/train.json \
    --dev ../data/dev.json \
    --output_dir ../checkpoints/ai_model \
    --epochs 5 \
    --batch_size 16 \
    --adversarial


### 6a. (Optional) Colab-safe checkpointing to Drive

Colab sessions can disconnect. Mount Drive and point `--output_dir` there so a
disconnect doesn't lose your fine-tuned weights:
```python
from google.colab import drive
drive.mount('/content/drive')
# then use: --output_dir /content/drive/MyDrive/ccc_project/checkpoints/ai_model
```

## 7. Run inference with the trained AI model and evaluate

`infer_ai.py` below is a small standalone script (not shipped in the original zip) that
loads the fine-tuned checkpoint and produces predictions in the same format
`evaluate.py` expects.

In [ ]:
infer_ai_code = r'''
import argparse, json, torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

def run(model_dir, data_path, out_path, max_length=128):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForTokenClassification.from_pretrained(model_dir).to(device).eval()
    id2label = model.config.id2label

    with open(data_path) as f:
        raw = json.load(f)
    items = raw if isinstance(raw, list) else list(raw.values())

    preds = []
    with torch.no_grad():
        for item in items:
            tokens = item["tokens"]
            enc = tok(tokens, is_split_into_words=True, truncation=True,
                       max_length=max_length, padding="max_length", return_tensors="pt")
            word_ids = enc.word_ids(batch_index=0)
            logits = model(input_ids=enc["input_ids"].to(device),
                            attention_mask=enc["attention_mask"].to(device)).logits[0]
            tag_ids = logits.argmax(-1).tolist()

            word_labels, prev_wid = [None] * len(tokens), None
            for pos, wid in enumerate(word_ids):
                if wid is None or wid == prev_wid:
                    continue
                word_labels[wid] = id2label[tag_ids[pos]]
                prev_wid = wid
            word_labels = [l if l is not None else "O" for l in word_labels]
            preds.append({"id": item.get("id"), "tokens": tokens, "pred_labels": word_labels})

    with open(out_path, "w") as f:
        json.dump(preds, f, indent=2)
    print(f"wrote {len(preds)} predictions to {out_path}")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_dir", default="../checkpoints/ai_model")
    ap.add_argument("--data", default="../data/test.json")
    ap.add_argument("--out", default="preds_ai_bert.json")
    args = ap.parse_args()
    run(args.model_dir, args.data, args.out)
'''
with open("infer_ai.py", "w") as f:
    f.write(infer_ai_code)
print("wrote infer_ai.py")


In [ ]:
!python infer_ai.py --model_dir ../checkpoints/ai_model --data ../data/test.json --out preds_ai_bert.json
!python evaluate.py --predictions preds_ai_bert.json --gold ../data/test.json --task ai


## 8. Train the AD (Acronym Disambiguation) model

DeepBlueAI-style candidate scoring — one forward pass per candidate long-form.

In [ ]:
!python train_ad_bert.py \
    --model_name bert-base-uncased \
    --train ../data/ad_train.json \
    --dev ../data/ad_dev.json \
    --output_dir ../checkpoints/ad_model \
    --epochs 3 \
    --batch_size 16


## 9. Run inference with the trained AD model and evaluate

`infer_ad.py` scores every candidate per acronym occurrence and picks the argmax,
writing `{"id":..., "pred_index":...}` rows for `evaluate.py --task ad`.

In [ ]:
infer_ad_code = r'''
import argparse, json, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

START_TOK, END_TOK = "[START]", "[END]"

def build_text(sentence, acronym_index):
    marked = sentence[:acronym_index] + [START_TOK, sentence[acronym_index], END_TOK] + sentence[acronym_index+1:]
    return " ".join(marked)

def run(model_dir, data_path, out_path, max_length=160):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()

    with open(data_path) as f:
        raw = json.load(f)
    items = raw if isinstance(raw, list) else list(raw.values())

    preds = []
    with torch.no_grad():
        for item in items:
            marked_text = build_text(list(item["sentence"]), item["acronym_index"])
            scores = []
            for cand in item["candidates"]:
                enc = tok(cand, marked_text, truncation=True, max_length=max_length,
                           padding="max_length", return_tensors="pt")
                logit = model(input_ids=enc["input_ids"].to(device),
                               attention_mask=enc["attention_mask"].to(device)).logits.squeeze(-1)
                scores.append(torch.sigmoid(logit).item())
            pred_index = int(torch.tensor(scores).argmax())
            preds.append({"id": item["id"], "pred_index": pred_index})

    with open(out_path, "w") as f:
        json.dump(preds, f, indent=2)
    print(f"wrote {len(preds)} predictions to {out_path}")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_dir", default="../checkpoints/ad_model")
    ap.add_argument("--data", default="../data/ad_test.json")
    ap.add_argument("--out", default="preds_ad_bert.json")
    args = ap.parse_args()
    run(args.model_dir, args.data, args.out)
'''
with open("infer_ad.py", "w") as f:
    f.write(infer_ad_code)
print("wrote infer_ad.py")


In [ ]:
!python infer_ad.py --model_dir ../checkpoints/ad_model --data ../data/ad_test.json --out preds_ad_bert.json
!python evaluate.py --predictions preds_ad_bert.json --gold ../data/ad_test.json --task ad


## 10. Save checkpoints + predictions back to Drive (recommended)

Colab wipes `/content` on disconnect. Copy anything you want to keep.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/ccc_project_results
# !cp -r ../checkpoints /content/drive/MyDrive/ccc_project_results/
# !cp preds_ai_bert.json preds_ad_bert.json /content/drive/MyDrive/ccc_project_results/


## Notes

- If you hit CUDA out-of-memory on `train_ai_bert.py`/`train_ad_bert.py`, lower
  `--batch_size` (try 8) or `--max_length` (try 96/128).
- Free-tier Colab GPUs (T4) can disconnect after ~90 min idle or ~12h total — keep the
  tab active, and checkpoint to Drive after each stage rather than only at the end.
- Next step once this all runs cleanly: swap `--model_name` to compare BERT vs SciBERT
  vs RoBERTa (Section 5 of the master guide), then try `--adversarial` on/off to
  quantify the AT-BERT-E-style gain yourself.